# `Dataset` types — a guide to every `dataset_type`

Once you have a `Fly`, `Experiment`, or list of either, `Dataset`
turns it into a tidy pandas DataFrame. The shape of that DataFrame is
controlled by one argument — `dataset_type` — and each option exists
to feed a specific kind of downstream analysis.

This notebook is a cookbook with:

- what each `dataset_type` gives you (schema + one-row example),
- what *requirements* it imposes (experiment type, skeleton track,
  Config flags), and
- a worked code snippet against a real fly.

Prerequisites: you've read
[`ballpushing_utils_walkthrough.ipynb`](./ballpushing_utils_walkthrough.ipynb)
so the `Fly` / `Experiment` / `Dataset` vocabulary is already familiar.

## Quick reference — dataset types at a glance

| `dataset_type`            | one row per …               | requires                                  | main use |
| ------------------------- | --------------------------- | ----------------------------------------- | -------- |
| `"summary"`               | (fly, ball)                 | nothing                                   | box/bar/permutation plots |
| `"coordinates"`           | (fly, frame)                | nothing                                   | trajectories, velocity profiles |
| `"fly_positions"`         | (fly, frame, keypoint)      | nothing                                   | heatmaps, keypoint analyses |
| `"event_metrics"`         | interaction event           | nothing                                   | per-event stats |
| `"F1_coordinates"`        | (fly, frame), F1-aware      | `experiment_type="F1"`                    | Fig 2 training/test traces |
| `"F1_checkpoints"`        | F1 milestone                | `experiment_type="F1"`                    | Fig 2 pretraining ladder |
| `"contact_data"`          | skeleton-contact frame      | `*_full_body.h5` skeleton track           | Fig 3 contact timelines |
| `"Skeleton_contacts"`     | skeleton contact            | `*_full_body.h5` skeleton track           | summary across contacts |
| `"standardized_contacts"` | standardized contact window | `*_full_body.h5` skeleton track           | cross-fly contact comparison |
| `"transformed"`           | event × flattened features  | `*_full_body.h5` skeleton track           | UMAP / ML feature table |
| `"transposed"`            | event-frame flat            | `*_full_body.h5` skeleton track           | frame-level ML |
| `"behavior_umap"`         | UMAP embedding row          | `transformed` or `transposed` upstream    | Behaviour UMAP panel |

For `experiment_type="Learning"`, any frame-level type (`coordinates`,
`fly_positions`, `standardized_contacts`) additionally gains `trial`
/ `trial_frame` / `trial_time` columns for free.

The rest of this notebook walks through these types from cheapest to
most-specialised.

## 0 · Setup

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

from ballpushing_utils import Config, Dataset, Experiment, Fly, data_root, load_dotenv

load_dotenv(Path.cwd().parent / ".env")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 60)

DATA_ROOT = data_root()
FLY_PATH = Path(os.environ.get(
    "BALLPUSHING_WALKTHROUGH_FLY",
    DATA_ROOT
        / "Ballpushing_Exploration/Videos/230704_FeedingState_1_AM_Videos_Tracked"
        / "arena2" / "corridor5",
))
# An F1-paradigm fly can be pointed at via a second env var. The
# F1-specific sections fall back to (skipped) when this isn't set.
F1_FLY_PATH = Path(os.environ.get(
    "BALLPUSHING_WALKTHROUGH_F1_FLY",
    DATA_ROOT / "F1_Tracks/Videos/__replace_with_an_F1_fly__" / "arena1" / "corridor1",
))

fly = Fly(FLY_PATH, as_individual=True) if FLY_PATH.is_dir() else None
if fly is None:
    print(f"⚠  Standard fly not reachable — data cells will be skipped.")
    print(f"   Tried: {FLY_PATH}")
else:
    print(f"Standard fly loaded: {fly.metadata.name}")

f1_fly: Fly | None = None
if F1_FLY_PATH.is_dir():
    # Force experiment_type="F1" for the F1-specific dataset types.
    f1_fly = Fly(F1_FLY_PATH, as_individual=True,
                 custom_config={"experiment_type": "F1"})
    print(f"F1 fly loaded:       {f1_fly.metadata.name}")
else:
    print(f"ℹ  No F1 fly configured — § F1 cells will be skipped.")
    print(f"   Set BALLPUSHING_WALKTHROUGH_F1_FLY to a corridor folder from an F1 experiment.")

### Helper — show the schema + a row preview

In [ ]:
def preview(ds: Dataset, max_cols: int = 10, rows: int = 3) -> None:
    """Print the shape, the first `max_cols` columns, and a few rows."""
    df = ds.data
    print(f"shape = {df.shape}")
    if df.empty:
        print("(empty DataFrame)")
        return
    print(f"first {min(max_cols, len(df.columns))} of {len(df.columns)} columns:")
    display(df.iloc[:rows, :max_cols])

## 1 · `"summary"` — one row per (fly, ball)

The default for figure pipelines. Every metric described in
[`ballpushing_metrics_reference.ipynb`](./ballpushing_metrics_reference.ipynb)
becomes a column, plus arena metadata (`Genotype`, `nickname`,
`brain_region`, `Light`, `FeedingState`, …).

Use this when you want to plot one point per fly and compare groups.

In [ ]:
if fly is None:
    print("(skipped)")
else:
    ds = Dataset(fly, dataset_type="summary")
    preview(ds)

## 2 · `"coordinates"` — one row per frame, fly + ball only

The standard per-frame view: centred fly/ball coordinates (origin =
fly start position), Euclidean distances, and an `interaction_event`
column that annotates which event (if any) each frame belongs to.

**Config levers that affect this type:**

- `downsampling_factor` (seconds) — keep every Nth-second frame.
- `annotate_contacts` — adds a `contact_event` column on top of
  `interaction_event` (requires skeleton track).
- `experiment_type="Learning"` — adds `trial` / `trial_time`.
- F1 flies automatically get an `adjusted_time` column (NaN for non-F1).

In [ ]:
if fly is None:
    print("(skipped)")
else:
    ds = Dataset(fly, dataset_type="coordinates")
    preview(ds, max_cols=12)
    print("\nInteraction-event-annotated frames:",
          int(ds.data["interaction_event"].notna().sum()))

## 3 · `"fly_positions"` — one row per frame, all keypoints

Same row cardinality as `coordinates`, but instead of just the thorax,
you get every column from the fly, ball, **and skeleton** tracks
(each suffixed with `_fly_<i>`, `_ball_<i>`, `_skeleton`). Useful for
heatmaps of legs/wings, or for any analysis that needs more than the
thorax position.

The skeleton columns only appear if a `*_full_body.h5` track exists.
Otherwise the dataset degrades gracefully to just fly + ball.

In [ ]:
if fly is None:
    print("(skipped)")
else:
    ds = Dataset(fly, dataset_type="fly_positions")
    preview(ds, max_cols=10)
    has_skeleton = any("_skeleton" in c for c in ds.data.columns)
    print(f"\nSkeleton columns present? {has_skeleton}")

## 4 · `"event_metrics"` — one row per interaction event

Zooms in from (fly, ball) aggregates to **per-event** metrics: start /
end frame, duration, event direction, back-off intervals, per-event
velocity. This is what `fly.event_metrics` exposes, flattened into a
DataFrame with arena metadata attached.

Use it when you want to correlate two metrics *within a fly* across
its own events (e.g. "did velocity drop after the major event?").

In [ ]:
if fly is None:
    print("(skipped)")
else:
    ds = Dataset(fly, dataset_type="event_metrics")
    preview(ds, max_cols=12)

## 5 · F1-paradigm datasets

The F1 paradigm has two corridors: a training ball (first corridor)
and a test ball (second corridor). `adjusted_time` is negative while
the fly is still in the training corridor and ≥ 0 after it exits.
These datasets encode that geometry explicitly.

**Precondition:** `fly.config.experiment_type == "F1"`. Without this,
`fly.f1_metrics` is `None` and the dataset builders return an empty
DataFrame. Set it via `custom_config={"experiment_type": "F1"}` or
`fly.config.experiment_type = "F1"`.

### 5.1 · `"F1_coordinates"` — training + test phases in one frame axis

Like `coordinates`, but with:

- `phase` column — `"training"` before the fly exits, `"test"` after.
- `phase_time` column — local time within each phase.
- `training_ball_*` / `test_ball_*` coordinate columns.
- 1-hour training-phase cutoff applied at dataset level (irrelevant
  if the fly's total time_range is already bounded).

Flies that never leave the training corridor produce an empty
DataFrame (with a warning).

In [ ]:
if f1_fly is None:
    print("(skipped — no F1 fly configured)")
else:
    ds = Dataset(f1_fly, dataset_type="F1_coordinates")
    preview(ds, max_cols=12)

### 5.2 · `"F1_checkpoints"` — one row per F1 milestone

Compact milestone table: for each pre-declared F1 checkpoint (e.g.
reaching 25 %, 50 %, 75 % of the test corridor), one row with the
adjusted time and the fly's exit time. Used to build the paper's
pretraining-ladder panels (Fig 2).

Returns an empty DataFrame when the fly never exits the corridor —
it literally has no checkpoints to record.

In [ ]:
if f1_fly is None:
    print("(skipped — no F1 fly configured)")
else:
    ds = Dataset(f1_fly, dataset_type="F1_checkpoints")
    preview(ds)

## 6 · Skeleton / contact datasets

These require the `*_full_body.h5` skeleton track. Without it,
`fly.skeleton_metrics` is `None` and the dataset builders return
empty frames.

If you want to enable contact annotations on the `coordinates` /
`fly_positions` datasets too, set `fly.config.annotate_contacts =
True`.

### 6.1 · `"contact_data"` — one row per frame **inside** a contact

Fine-grained contact view: every frame that falls inside a detected
skeleton-contact window becomes a row, carrying that frame's skeleton
keypoints + ball position + a `contact_index` identifying which
contact it belongs to. Missing keypoints are filled with
`Config.hidden_value` (default -9999).

Respects `Config.success_cutoff` — when enabled, contacts after the
final success event are dropped.

In [ ]:
if fly is None:
    print("(skipped)")
elif fly.skeleton_metrics is None:
    print("This fly has no skeleton track (*_full_body.h5) — contact datasets will be empty.")
else:
    ds = Dataset(fly, dataset_type="contact_data")
    preview(ds, max_cols=10)

### 6.2 · `"Skeleton_contacts"` — one row per contact

One row per detected contact event with the matching ball
displacement. Minimal schema: `contact_index`, `ball_displacement`,
plus arena metadata. Handy for histograms / survival curves of
contact effectiveness.

In [ ]:
if fly is None or fly.skeleton_metrics is None:
    print("(skipped)")
else:
    ds = Dataset(fly, dataset_type="Skeleton_contacts")
    preview(ds)

### 6.3 · `"standardized_contacts"` — resampled contact windows

One row per frame, but contacts are **time-standardised** (same number
of frames per contact across flies), so you can stack them into a
uniform tensor. This is the input format for the behaviour UMAP
analyses.

In [ ]:
if fly is None or fly.skeleton_metrics is None:
    print("(skipped)")
else:
    ds = Dataset(fly, dataset_type="standardized_contacts")
    preview(ds, max_cols=10)

## 7 · ML feature tables — `"transformed"`, `"transposed"`, `"behavior_umap"`

These three form the pipeline that feeds the behaviour UMAP panels in
the paper.

- `"transposed"` — one row per (event, frame) with all keypoint
  features flattened. The rawest of the ML-oriented views.
- `"transformed"` — one row per event; per-frame features from
  `standardized_contacts` are flattened into fixed-length vectors plus
  statistical summaries (Fourier, per-keypoint moments, etc.) and
  joined with the per-event metrics from `event_metrics`. This is
  what you feed to scikit-learn.
- `"behavior_umap"` — runs a pre-configured UMAP on the feature table
  and returns the embedding as a DataFrame, ready to scatter-plot.

All three require the skeleton track.

In [ ]:
if fly is None or fly.skeleton_metrics is None:
    print("(skipped)")
else:
    ds = Dataset(fly, dataset_type="transformed")
    print("`transformed` shape:", ds.data.shape)
    if not ds.data.empty:
        # Just show which families of columns exist — the full width is huge.
        prefixes = sorted({c.split("_")[0] for c in ds.data.columns})
        print("Column prefixes:", prefixes[:20])

## 8 · Learning-paradigm awareness

When `fly.config.experiment_type == "Learning"`, **any** frame-level
dataset (`coordinates`, `fly_positions`, `standardized_contacts`,
etc.) automatically gains three columns:

- `trial` — integer trial index (0 outside any detected trial),
- `trial_frame` — frame counter reset to 0 at each trial's start,
- `trial_time` — `trial_frame / fps` in seconds.

There are no `Dataset(dataset_type="learning_…")` flavours — the
metrics for the learning paradigm (trial durations, peak heights,
etc.) live under `fly.learning_metrics` and flow into the regular
`"summary"` dataset when the experiment type matches.

If you want trial-aware frame data for a non-Learning recording, set
the type and construct the fly explicitly:

```python
fly = Fly(path, as_individual=True,
          custom_config={"experiment_type": "Learning"})
ds = Dataset(fly, dataset_type="coordinates")
# ds.data now has trial / trial_frame / trial_time columns.
```

## 9 · Pooling across flies and experiments

The `source` argument accepts more than a single `Fly`:

```python
# Pool every fly in one experiment folder
Dataset(Experiment("/path/to/experiment_dir"), dataset_type="summary")

# Pool multiple experiments (recording days)
Dataset([exp_a, exp_b, exp_c], dataset_type="summary")

# Hand-pick flies across experiments
flies = exp_a.find_flies("metadata.nickname", "PR") + \
        exp_b.find_flies("metadata.nickname", "PR")
Dataset(flies, dataset_type="summary")
```

In every case the output is a single DataFrame with one row per
(fly, ball) (or per frame / per event, depending on `dataset_type`),
and every row is tagged with the full arena metadata via
`_add_metadata`, so you can `groupby` without re-joining anything.

For production pipelines, pre-computed pooled datasets are usually
persisted as `.feather` files; see `figures/*` for examples that
consume them.

## 10 · Cheat-sheet — which `dataset_type` should I pick?

| I want to …                                    | Use |
| ---------------------------------------------- | --- |
| Compare one metric across genotypes            | `"summary"` |
| Plot a trajectory or velocity profile          | `"coordinates"` |
| Build a spatial heatmap with all keypoints     | `"fly_positions"` |
| Analyse one event at a time (not one fly)      | `"event_metrics"` |
| Show training vs test F1 phases on one axis    | `"F1_coordinates"` |
| Rank flies by F1 milestone timing              | `"F1_checkpoints"` |
| Look at leg-by-leg contact kinematics          | `"contact_data"` |
| Histogram ball displacement per contact        | `"Skeleton_contacts"` |
| Build a per-event feature table for ML         | `"transformed"` |
| Scatter the UMAP embedding                     | `"behavior_umap"` |

And two recurring traps:

1. **Empty DataFrame back?** The builder printed a warning — read it.
   Common causes: missing skeleton track, F1 type not set,
   `success_cutoff` chopped off everything interesting.
2. **Missing metric column in `"summary"`?** Check
   `fly.config.enabled_metrics`; the default set excludes binned /
   R² / logistic fits to keep computation cheap. Set it to `None` to
   compute everything.